In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Pré traitement des données 

In [2]:
df = pd.read_csv('data/projects_data.csv')

In [3]:
df.head()

,project_name,link,description,composants
0,Tillu - the Robot,https://www.instructables.com/Tillu-the-Robot,"MeetTillu - The Robot, a unique fusion of adva...",1xUnihiker 4xServo 1xBattery Managment 2xUSB T...
1,Air Hockey Robot With Shot Prediction,https://www.instructables.com/Air-Hockey-Robot,Ever wanted to play air hockey against a compu...,Disclaimer - Supplies list may include affilia...
2,Two-Wheeled Drive Car With a Robotic Arm,https://www.instructables.com/Two-Wheeled-Driv...,This project was developed within theOrange Di...,To start building a Two-Wheeled Drive Car with...
3,Ghost Candy Dispenser,https://www.instructables.com/Ghost-Candy-Disp...,Hi there! Did you enjoy Halloween in 2024? Sin...,Parts MCU:M5StampS3 (M5Stack)(×1) RC servo mot...
4,Lily∞Bot With Micro:Bit and Motor:Bit Does Obs...,https://www.instructables.com/LilyBot-With-Mic...,This project will explain how to get obstacle ...,The parts list for the Lily∞Bot With Micro:Bit...


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1503 entries, 0 to 1502
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   project_name  1499 non-null   object
 1   link          1503 non-null   object
 2   description   1178 non-null   object
 3   composants    1169 non-null   object
dtypes: object(4)
memory usage: 47.1+ KB


In [5]:
df.isna().sum()

project_name      4
link              0
description     325
composants      334
dtype: int64

On a:
- *325* projets sans descriptions
- *334* projets dont les composants pour la concéption ne sont pas indiqués.

Je vais tous simplement supprimer les lignes sans l'un de ces attributs car on ne pourra pas récommender à une personne un projet si on ne sait même pas de quoi parle le projet...

In [6]:
df_cleaned = df.dropna(subset=['composants','description'])
df_cleaned.isnull().sum()

project_name    0
link            0
description     0
composants      0
dtype: int64

**Je vais traduire les textes en Français pour la suite de l'analyse**

In [7]:
from transformers import MarianMTModel, MarianTokenizer

/Users/amadouu/M2_ML/BD/env_bigData/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/usr/local/Cellar/python@3.12/3.12.6/Frameworks/Python.framework/Versions/3.12/lib/python3.12/runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/local/Cellar/python@3.12/3.12.6/Frameworks/Python.fr

In [10]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast

model = MBartForConditionalGeneration.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")
tokenizer = MBart50TokenizerFast.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")


In [ ]:
def translate_text(text):
    translated = model.generate(**text)
    translated_text = tokenizer.decode(translated[0], skip_special_tokens=True)
    translated = model.generate(**tokenizer(src_text, return_tensors="pt", padding=True))
    [tokenizer.decode(t, skip_special_tokens=True) for t in translated] 
    return " ".join(translations)

In [131]:
df_cleaned['composants_fr'] = df_cleaned['composants'].apply(lambda x: translator.translate(x))

NotValidLength: The majority of our parts have been sourced from GoBilda, a proud materials provider for FTC robotics. Their parts are quite high quality, but also reasonably expensive. I won't provide every single structure, as there are too many individual pieces to count, but I'll list the major components, such as the control hub, driver and expansion hubs, and major pieces.  Off-the-Shelf Parts List: 1x REV Robotics Control Hub- $350.00 - https://www.revrobotics.com/rev-31-1595/ 1x REV Robotics Expansion Hub- $250.00 - https://www.revrobotics.com/rev-31-1153/ The Control and Expansion Hub is where all your programs and core information will be stored, as well as where all your wiring is connected to. 6x GoBilda Yellow Jacket 19:2:1 Gear Motors (Planetary, REX Bore)- $42.99/pc - https://www.gobilda.com/5203-series-yellow-jacket-planetary-gear-motor-19-2-1-ratio-24mm-length-8mm-rex-shaft-312-rpm-3-3-5v-encoder/ 6x GoBilda Yellow Jacket 5:2:1 Gear Motors (Planetary, REX Bore)- $42.99